# AI Signal Processing Lab

This notebook calls the shared `signal_lab` package.

Processing functions live in the package modules.
This notebook contains explanations, settings, and visualisations.

Run cells from top to bottom.
The project folder must be available before running setup.


In [ ]:
# Locate the project containing pyproject.toml.
from pathlib import Path
import subprocess
import sys

project_root = Path("/content/ai_signal_lab")

# Stop clearly if the project has not been restored to this runtime.
if not (project_root / "pyproject.toml").is_file():
    raise FileNotFoundError(
        "Restore the ai_signal_lab project folder under /content first."
    )

# Install the shared package using this notebook's Python interpreter.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(project_root)],
    check=True,
)

# Import numerical tools and the packaged processing functions.
import numpy as np
from signal_lab.validation import validate_signal
from signal_lab.spectral import calculate_fft, detect_peaks
from signal_lab.recommendation import recommend_filter
from signal_lab.filters import apply_filter
from signal_lab.metrics import compare_rms
from signal_lab.data_io import load_wav, load_csv, save_processed_wav

print("Shared processing functions imported successfully.")

## Demonstration: identify and remove an unwanted tone

The example contains a useful 100 Hz tone and an unwanted 400 Hz tone.
Sampling rate: 2000 Hz. Duration: 5 seconds.
Amplitudes are dimensionless.

We declare 80–250 Hz as the useful band.
The recommendation treats detected peaks outside that band as unwanted.
This is a rule based on our declared band, not general noise recognition.

In [ ]:
# Set the sampling rate in samples per second (Hz).
fs_hz = 2000.0

# Set the recording duration in seconds.
duration_s = 5.0

# Generate one time value for each sample, in seconds.
time_s = np.arange(int(fs_hz * duration_s)) / fs_hz

# Create the useful 100 Hz tone with amplitude 1.
useful = np.sin(2 * np.pi * 100.0 * time_s)

# Create the unwanted 400 Hz tone with amplitude 0.5.
unwanted = 0.5 * np.sin(2 * np.pi * 400.0 * time_s)

# Combine the tones and validate using the shared package.
original = validate_signal(useful + unwanted, fs_hz)

# Record amplitude units for later plots.
amplitude_unit = "dimensionless"

# Display basic recording information.
print("Sampling rate:", fs_hz, "Hz")
print("Duration:", len(original) / fs_hz, "seconds")
print("Nyquist frequency:", fs_hz / 2.0, "Hz")

In [ ]:
# Leave this as None to use the demonstration signal.
# For your file, replace None with a path such as "/content/my_audio.wav".
input_path = None

# Select a WAV channel; 0 means the first channel.
wav_channel = 0

# Supply a rate in Hz for CSV files without time_s.
# Leave None when the CSV contains timestamps.
csv_fs_hz = None

# State your CSV amplitude units, such as "V" or "dimensionless".
csv_amplitude_unit = "V"

# The demonstration is digital audio and can be exported as WAV.
export_audio = True

# Replace the demonstration only when an input file is selected.
if input_path is not None:
    selected_path = Path(input_path)

    # Stop clearly if the selected file is missing.
    if not selected_path.is_file():
        raise FileNotFoundError(f"Input file not found: {selected_path}")

    # Identify the file type from its extension.
    extension = selected_path.suffix.lower()

    # Load WAV audio through the shared package.
    if extension == ".wav":
        original, fs_hz = load_wav(selected_path, channel=wav_channel)
        amplitude_unit = "dimensionless"

    # Load CSV measurements through the shared package.
    elif extension == ".csv":
        original, fs_hz = load_csv(selected_path, fs_hz=csv_fs_hz)
        amplitude_unit = csv_amplitude_unit

        # Only explicitly declared digital audio can be exported as WAV.
        export_audio = amplitude_unit == "dimensionless"

    # Reject file types that our lab does not support.
    else:
        raise ValueError("Select a .wav or .csv file.")

    # Display information for the selected recording.
    print("Input file:", selected_path.name)
    print("Sampling rate:", fs_hz, "Hz")
    print("Duration:", len(original) / fs_hz, "seconds")
    print("Nyquist frequency:", fs_hz / 2.0, "Hz")
    print("Amplitude units:", amplitude_unit)
else:
    print("Using the demonstration signal.")

In [ ]:
# Calculate the spectrum and detect prominent peaks.
frequencies_hz, amplitudes, peak_indices = detect_peaks(
    original, fs_hz
)

# Extract the frequencies at the detected peak positions.
detected_hz = frequencies_hz[peak_indices]
print("Detected frequencies (Hz):", detected_hz)

# Declare the useful frequency band, in Hz.
useful_low_hz = 80.0
useful_high_hz = 250.0

# Request a filter recommendation from the shared module.
choice = recommend_filter(
    detected_hz,
    fs_hz,
    useful_low_hz=useful_low_hz,
    useful_high_hz=useful_high_hz,
)
print("Recommendation:", choice)

# Apply the recommended filter through the shared module.
filtered = apply_filter(
    original,
    fs_hz,
    filter_type=choice["filter_type"],
    cutoff_hz=choice["cutoff_hz"],
)

# Compare RMS amplitudes in the original signal's units.
print("RMS comparison:", compare_rms(original, filtered))

## Compare before and after

The waveform plot shows the first 0.05 seconds.
The spectrum overlay shows the frequency content of the whole recording.

For this example, the filtered 400 Hz peak should shrink substantially,
while the 100 Hz peak should remain close to its original amplitude.

Both spectra use the same amplitude scale.
A lower RMS alone does not prove improved signal quality.

In [ ]:
# Import the plotting library.
import matplotlib.pyplot as plt

# Calculate both spectra using the same packaged FFT function.
before_hz, before_amplitude = calculate_fft(original, fs_hz)
after_hz, after_amplitude = calculate_fft(filtered, fs_hz)

# Construct sample times in seconds from the actual signal length.
plot_time_s = np.arange(len(original)) / fs_hz

# Select the first 0.05 seconds for a readable waveform plot.
visible = plot_time_s < 0.05

# Create two vertically arranged plots.
figure, axes = plt.subplots(2, 1, figsize=(10, 7))

# Overlay the input and output waveforms without rescaling them.
axes[0].plot(
    plot_time_s[visible], original[visible],
    label="Before", alpha=0.7,
)
axes[0].plot(
    plot_time_s[visible], filtered[visible],
    label="After", linewidth=2,
)

# Label the waveform plot with its units.
axes[0].set_title("Waveform comparison")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel(f"Amplitude ({amplitude_unit})")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Overlay the spectra on the same axes and amplitude scale.
axes[1].plot(before_hz, before_amplitude, label="Before", alpha=0.7)
axes[1].plot(after_hz, after_amplitude, label="After")

# Shade the useful frequency band declared earlier.
axes[1].axvspan(
    useful_low_hz, useful_high_hz,
    color="green", alpha=0.1, label="Declared useful band",
)

# Display up to 600 Hz, or Nyquist if it is lower.
axes[1].set_xlim(0, min(600.0, fs_hz / 2.0))

# Label the spectrum plot with frequency and amplitude units.
axes[1].set_title("Spectrum comparison")
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel(f"Amplitude ({amplitude_unit})")
axes[1].legend()
axes[1].grid(alpha=0.3)

# Adjust spacing and display both plots.
figure.tight_layout()
plt.show()

## Save and download processed audio

This cell saves the filtered signal using the shared package.

WAV export expects dimensionless digital audio.
Signals with peaks above 1 are scaled down to prevent overflow.
The amplitude divisor is reported below.

Set download_audio = True when running interactively in Google Colab
to download the saved WAV. Leave it False for automated execution.

In [ ]:
if export_audio:
    # Only export signals explicitly labelled as digital audio.
    if amplitude_unit != "dimensionless":
        raise ValueError(
            "Convert measurement amplitudes to digital audio before WAV export."
        )

    # Create a folder for generated output files.
    output_folder = project_root / "outputs"
    output_folder.mkdir(parents=True, exist_ok=True)

    # Choose a filename distinct from the input file.
    output_path = output_folder / "processed_audio.wav"

    # Save using the packaged export function.
    amplitude_divisor = save_processed_wav(
        output_path, filtered, fs_hz
    )

    # Display the file location and any amplitude scaling.
    print("Saved:", output_path)
    print("Amplitude divisor:", amplitude_divisor)

    # Keep browser downloading off during automated notebook checks.
    download_audio = False

    # Enable this option to download from an interactive Colab session.
    if download_audio:
        from google.colab import files
        files.download(str(output_path))
else:
    print("WAV export skipped: input is measurement data.")
